# 美股组合回测 Demo

本 Notebook 演示使用 `vectorbt_qs` 进行美股多空回测的完整流程：

1. 加载行情数据（PanelDaily）
2. 生成动量多空策略的目标权重
3. 应用美股交易约束（SSR）
4. 执行回测并对比多空 vs 纯做多

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__")))))

from mvp.data.adapter import load_us_daily_bar, load_us_dims
from mvp.engine.runner import run_backtest, portfolio_report, compare_reports
import vectorbt as vbt

print("✅ 环境就绪")

In [ ]:
# ======== 可调参数 ========
MARKET = "us"
START = "2024-06-01"             # ← 1 年数据
END = "2025-06-01"
INIT_CASH = 1_000_000.0
MAX_SYMBOLS = 30
WEIGHT_PER_STOCK = 0.05

# 数据源: "cos" = COS 直接读 | "local" = 本地 data/
DATA_SOURCE = "cos"

BACKTEST_CONFIG = {
    "init_cash": INIT_CASH,
    "slippage": 0.0005,
    "freq": "1D",
}

# 切换数据源
if DATA_SOURCE == "cos":
    from mvp.data.adapter import set_data_root
    set_data_root("us_stock", "cos://qs-cold/clean_data/us_stock/massive_data")
    print("⚠️  首次从 COS 读取会下载并缓存 (~3-8 分钟/年)")
else:
    print("📂 使用本地 data/ 目录")

print(f"📅 回测区间: {START} ~ {END}")
print(f"💰 初始资金: ${INIT_CASH:,.0f}")
print(f"📊 最多标的: {MAX_SYMBOLS}")

In [ ]:
# 加载数据（PanelDaily 含复权价 + 收益率 + 基本面 PIT）
data = load_us_daily_bar(symbols=None, start=START, end=END, use_panel=True)
close = data["close"]

# 限制标的数量
if MAX_SYMBOLS and close.shape[1] > MAX_SYMBOLS:
    top_symbols = close.iloc[-1].dropna().sort_values(ascending=False).head(MAX_SYMBOLS).index
    close = close[top_symbols]

print(f"📊 数据形状: {close.shape[0]} 天 × {close.shape[1]} 只")
print(f"📅 日期范围: {close.index[0].date()} ~ {close.index[-1].date()}")
print(f"\n前 5 只标的: {list(close.columns[:5])}")
close.head()

In [ ]:
# ======== 选择权重生成模式 ========
WEIGHT_MODE = "random"  # 'random' | 'index_like'

def random_long_short(close, n_long=8, n_short=4, weight=0.05, seed=42):
    """每天随机多空"""
    rng = np.random.default_rng(seed)
    weights = pd.DataFrame(0.0, index=close.index, columns=close.columns)
    all_c = close.columns.tolist()
    for i, date in enumerate(close.index):
        shuffled = rng.permutation(all_c)
        n = n_long + n_short
        chosen = shuffled[:min(n, len(all_c))]
        longs = chosen[:n_long]
        shorts = chosen[n_long:n_long+n_short] if len(chosen) > n_long else []
        weights.loc[date, longs] = weight
        weights.loc[date, shorts] = -weight
    return weights

def random_long_only(close, n_stocks=12, weight=0.05, seed=99):
    """每天随机纯做多"""
    rng = np.random.default_rng(seed)
    weights = pd.DataFrame(0.0, index=close.index, columns=close.columns)
    for i, date in enumerate(close.index):
        chosen = rng.choice(close.columns, size=min(n_stocks, len(close.columns)), replace=False)
        weights.loc[date, chosen] = weight
    return weights

def index_like_ls(close, n_long=10, n_short=3, noise_std=0.01, turnover=0.15, weight=0.05, seed=42):
    """指数型多空：低换手 + 噪声"""
    rng = np.random.default_rng(seed)
    weights = pd.DataFrame(0.0, index=close.index, columns=close.columns)
    all_c = close.columns.tolist()
    long_pool = rng.choice(all_c, size=n_long, replace=False).tolist()
    remaining = [c for c in all_c if c not in long_pool]
    short_pool = rng.choice(remaining, size=n_short, replace=False).tolist()
    
    for i, date in enumerate(close.index):
        if rng.random() < turnover or i == 0:
            long_pool = rng.choice(all_c, size=n_long, replace=False).tolist()
            remaining = [c for c in all_c if c not in long_pool]
            short_pool = rng.choice(remaining, size=n_short, replace=False).tolist() if len(remaining) >= n_short else []
        for t in long_pool:
            weights.loc[date, t] = max(0, weight + rng.normal(0, noise_std))
        for t in short_pool:
            weights.loc[date, t] = min(0, -weight + rng.normal(0, noise_std))
    return weights

# 生成两组
if WEIGHT_MODE == "random":
    target_weights_ls = random_long_short(close)
    target_weights_lo = random_long_only(close)
else:
    target_weights_ls = index_like_ls(close)
    target_weights_lo = random_long_only(close)

for name, tw in [("多空", target_weights_ls), ("纯做多", target_weights_lo)]:
    l = (tw > 0).sum(axis=1).mean()
    s = (tw < 0).sum(axis=1).mean()
    print(f"📈 {name}: 日均多 {l:.0f} 只, 日均空 {s:.0f} 只")

In [ ]:
# 多空策略回测
print("── 多空策略 ──")
pf_ls = run_backtest(MARKET, target_weights_ls, config=BACKTEST_CONFIG)

# 纯做多回测
print("\n── 纯做多策略 ──")
pf_lo = run_backtest(MARKET, target_weights_lo, config=BACKTEST_CONFIG)

print("\n✅ 回测完成")
print(f"   多空: {len(pf_ls.trades.records_readable)} 笔交易, 净值 ${pf_ls.final_value():,.0f}")
print(f"   做多: {len(pf_lo.trades.records_readable)} 笔交易, 净值 ${pf_lo.final_value():,.0f}")

# 策略对比
comparison = compare_reports({"多空随机": pf_ls, "纯做多随机": pf_lo})
comparison

In [ ]:
# 策略对比
results = {"动量多空": pf_ls, "动量纯做多": pf_lo}
comparison = compare_reports(results)
comparison

In [ ]:
# 净值对比
import plotly.graph_objects as go

fig = go.Figure()
for label, pf_obj in results.items():
    value = pf_obj.value()
    fig.add_trace(go.Scatter(
        x=value.index, y=value.values.flatten(),
        mode='lines', name=label
    ))
fig.update_layout(
    title="美股组合净值对比 — 动量策略",
    xaxis_title="日期",
    yaxis_title="净值 ($)",
    hovermode='x unified'
)
fig.show()

In [ ]:
# 多空策略的 trades 明细
trades = pf_ls.trades.records_readable
if len(trades) > 0:
    long_trades = trades[trades["Direction"] == "Long"]
    short_trades = trades[trades["Direction"] == "Short"]
    
    print(f"📊 多空策略交易明细:")
    print(f"   做多交易: {len(long_trades)} 笔, 胜率: {long_trades['Return'].gt(0).mean():.1%}")
    print(f"   做空交易: {len(short_trades)} 笔, 胜率: {short_trades['Return'].gt(0).mean():.1%}")
    print(f"\n   最佳多: PnL={long_trades['PnL'].max():.0f}, 最佳空: PnL={short_trades['PnL'].max():.0f}")
    print(f"   最差多: PnL={long_trades['PnL'].min():.0f}, 最差空: PnL={short_trades['PnL'].min():.0f}")
else:
    print("⚠️ 无成交记录")

## 总结

本 Demo 展示了：
- 动量多空 vs 纯做多的绩效对比
- `run_backtest` 一行接入美股约束（SSR）
- `compare_reports` 快速对比多组策略
- `trades.records_readable` 获取逐笔交易明细

> 💡 **替换策略**：只需修改 `momentum_long_short` 函数，产出新的 `target_weights`，即可回测你的自定义策略。